<a href="https://colab.research.google.com/github/Prakashbhaskar123/DeviceSymmetryAnalysis-n_and_pGAAFET-/blob/main/BSIM_ModelTraining.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
import os
import torch
import pandas as pd
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn

drive.mount('/content/drive')

# Define your exact folder paths
tcad_dir = '/content/drive/MyDrive/BSIMPrametersANNModel/idvgTCAD_DATA'
mystic_dir = '/content/drive/MyDrive/BSIMPrametersANNModel/mystic/stage1'

Mounted at /content/drive


In [6]:
class GAAFET_Dataset(Dataset):
    def __init__(self, tcad_dir, mystic_dir, node_list):
        self.tcad_dir = tcad_dir
        self.mystic_dir = mystic_dir
        self.node_list = node_list

    def __len__(self):
        return len(self.node_list)

    def parse_mod_file(self, filepath):
        """Extracts VTH0, CDSC, U0, and NFACTOR from the .mod file."""
        params = {'VTH0': 0.4, 'CDSC': 0.0, 'U0': 0.02, 'NFACTOR': 1.0}
        with open(filepath, 'r') as file:
            for line in file:
                if line.startswith('+'):
                    clean_line = line.replace('+', '').replace('=', ' ').strip()
                    tokens = clean_line.split()
                    for i in range(0, len(tokens), 2):
                        param_name = tokens[i].upper()
                        if param_name in params:
                            params[param_name] = float(tokens[i+1])

        return torch.tensor([params['VTH0'], params['CDSC'],
                             params['U0'], params['NFACTOR']], dtype=torch.float32)

    def __getitem__(self, idx):
        node = self.node_list[idx]

        # 1. Load Inputs (X)
        csv_path = os.path.join(self.tcad_dir, f"ML_TrainingData_n{node}.csv")
        df = pd.read_csv(csv_path)
        id_array = torch.tensor(df['drain TotalCurrent'].abs().values, dtype=torch.float32)
        x_tensor = torch.log10(id_array + 1e-15)

        # 2. Load Labels (Y)
        mod_path = os.path.join(self.mystic_dir, f"BSIM_Parameters_n{node}.mod")
        y_tensor = self.parse_mod_file(mod_path)

        return x_tensor, y_tensor